#  Pharma Demand Analysis — Feature_Engneering

---

##  Context

This dataset combines:

- Real pharmaceutical demand patterns (monthly prescription data)
- Simulated supply chain features (pharmacies, cities, stock, lead time)

The goal is to analyze demand behavior and understand patterns that will inform forecasting and supply decisions.

---

##  Objectives

- Understand demand distribution across drugs
- Identify seasonality and trends
- Analyze variability and spikes in demand
- Examine supply-demand gaps (stock vs demand)
- Compare demand across cities and pharmacies




# 1.0 Libraries

In [1]:
# Data tool kit
import pandas as pd
import numpy as np

# Visualization tool
import  matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings("ignore")

print("imported sucessfully")

imported sucessfully


# Loading Data

In [7]:
# 
df = pd.read_parquet("../data/processed/final_pharmalink_dataset.parquet")
df.head()

,date,drug,demand,cost,pharmacy_id,city,stock,lead_time_days
0,2021-01-01,Amoxicillin,309442,558748.88,1,Tier2,309418,4
1,2021-01-01,Amoxicillin,513312,558748.88,2,Tier2,513397,4
2,2021-01-01,Amoxicillin,435921,558748.88,3,Tier2,435955,2
3,2021-01-01,Amoxicillin,388742,558748.88,4,Tier2,388704,3
4,2021-01-01,Amoxicillin,232121,558748.88,5,Tier3,232128,2


# 2.0 Feature Engneering

## 2.1 Time-based features


In [8]:
# Ensure datetime
df['date'] = pd.to_datetime(df['date'])

# time based
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['quarter'] = df['date'].dt.quarter
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)

## 2.2 Sort for time-series ops

In [9]:
df = df.sort_values(by=['pharmacy_id', 'drug', 'date'])

## 2.3 Lag Features

df['demand_lag_1'] = df.groupby(['pharmacy_id', 'drug'])['demand'].shift(1)
df['demand_lag_7'] = df.groupby(['pharmacy_id', 'drug'])['demand'].shift(7)

## 2.4 Rolling Statistics

In [10]:
df['rolling_mean_7'] = df.groupby(['pharmacy_id', 'drug'])['demand'].transform(lambda x: x.shift(1).rolling(7).mean())
df['rolling_std_7'] = df.groupby(['pharmacy_id', 'drug'])['demand'].transform(lambda x: x.shift(1).rolling(7).std())


## 2.5 Stock Demand Features

In [11]:
df['stock_gap'] = df['stock'] - df['demand']
df['stock_ratio'] = df['stock'] / (df['demand'] + 1)
df['stockout_flag'] = (df['stock'] < df['demand']).astype(int)


## 2.6 Lead time features

In [12]:
df['demand_per_lead_time'] = df['demand'] / (df['lead_time_days'] + 1)
df['urgency_score'] = (df['demand'] / (df['stock'] + 1)) * df['lead_time_days']

## 2.7 Trend Features

In [14]:
df = df.sort_values(by=['pharmacy_id', 'drug', 'date'])

# 2. Group by the entities and shift the demand by 1 month to create the lag
df['demand_lag_1'] = df.groupby(['pharmacy_id', 'drug'])['demand'].shift(1)


df['demand_diff_1'] = df['demand'] - df['demand_lag_1']
df['demand_growth_rate'] = df['demand_diff_1'] / (df['demand_lag_1'] + 1)

df.head()


,date,drug,demand,cost,pharmacy_id,city,stock,lead_time_days,day_of_week,month,...,rolling_mean_7,rolling_std_7,stock_gap,stock_ratio,stockout_flag,demand_per_lead_time,urgency_score,demand_lag_1,demand_diff_1,demand_growth_rate
0,2021-01-01,Amoxicillin,309442,558748.88,1,Tier2,309418,4,4,1,...,NaN,NaN,-24,0.999919,1,61888.400000,4.000297,NaN,NaN,NaN
100,2021-02-01,Amoxicillin,148913,449788.26,1,Tier1,148888,4,0,2,...,NaN,NaN,-25,0.999825,1,29782.600000,4.000645,309442.0,-160529.0,-0.518768
200,2021-03-01,Amoxicillin,370401,525544.44,1,Tier3,370431,2,0,3,...,NaN,NaN,30,1.000078,0,123467.000000,1.999833,148913.0,221488.0,1.487355
300,2021-04-01,Amoxicillin,180832,557305.44,1,Tier2,180920,3,3,4,...,NaN,NaN,88,1.000481,0,45208.000000,2.998524,370401.0,-189569.0,-0.511793
400,2021-05-01,Amoxicillin,236468,674377.43,1,Tier2,236510,5,5,5,...,NaN,NaN,42,1.000173,0,39411.333333,4.999091,180832.0,55636.0,0.307665


## 2.8 Cost-based Features

In [15]:
df['revenue_estimate'] = df['demand'] * df['cost']
df['inventory_value'] = df['stock'] * df['cost']


## 2.9 City Encoding

In [16]:
df = pd.get_dummies(df, columns=['city'], drop_first=True)


## 2.10 Pharmacy-level aggregations

In [17]:
pharmacy_stats = df.groupby('pharmacy_id')['demand'].agg(['mean', 'std']).reset_index()
pharmacy_stats.columns = ['pharmacy_id', 'pharmacy_avg_demand', 'pharmacy_demand_std']

## 2.11. Interaction features

In [18]:
df['lead_time_stock_gap'] = df['lead_time_days'] * df['stock_gap']

## 2.12 Risk flags

In [19]:
df['high_risk_flag'] = ((df['stock'] < df['demand']) & (df['lead_time_days'] > 3)).astype(int)
df['overstock_flag'] = (df['stock'] > df['demand'] * 1.5).astype(int)


## 3.0 Final

In [20]:
df = df.fillna(0)

# Final dataframe ready
df.head()

,date,drug,demand,cost,pharmacy_id,stock,lead_time_days,day_of_week,month,week_of_year,...,demand_lag_1,demand_diff_1,demand_growth_rate,revenue_estimate,inventory_value,city_Tier2,city_Tier3,lead_time_stock_gap,high_risk_flag,overstock_flag
0,2021-01-01,Amoxicillin,309442,558748.88,1,309418,4,4,1,53,...,0.0,0.0,0.000000,1.729004e+11,1.728870e+11,True,False,-96,1,0
100,2021-02-01,Amoxicillin,148913,449788.26,1,148888,4,0,2,5,...,309442.0,-160529.0,-0.518768,6.697932e+10,6.696807e+10,False,False,-100,1,0
200,2021-03-01,Amoxicillin,370401,525544.44,1,370431,2,0,3,9,...,148913.0,221488.0,1.487355,1.946622e+11,1.946780e+11,False,True,60,0,0
300,2021-04-01,Amoxicillin,180832,557305.44,1,180920,3,3,4,13,...,370401.0,-189569.0,-0.511793,1.007787e+11,1.008277e+11,True,False,264,0,0
400,2021-05-01,Amoxicillin,236468,674377.43,1,236510,5,5,5,17,...,180832.0,55636.0,0.307665,1.594687e+11,1.594970e+11,True,False,210,0,0
